In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd

from sklearn.neighbors import BallTree
from math import radians

In [2]:
# worldpop

world_pop = pd.read_parquet("../data/03_primary/prm_world_pop")

In [3]:
world_pop.head()

,latitude,longitude,pop_density
0,39.294582,32.262085,2.210868
1,39.302917,32.253750,1.977098
2,39.302917,32.245419,4.719284
3,39.311249,32.245419,1.628673
4,39.302917,32.237083,4.009533


In [128]:
# OSM

In [24]:
prm_poi_df = gpd.read_file("../data/03_primary/saudi_arabia/osm/poi.geojson")
prm_highways_df = gpd.read_file("../data/03_primary/saudi_arabia/osm/highways.geojson")
prm_subway_df = gpd.read_file("../data/03_primary/saudi_arabia/osm/subway.geojson")

# prm_poi_df = gpd.read_file("../data/02_intermediate/saudi_arabia/osm/poi.geojson")
# prm_highways_df = gpd.read_file("../data/02_intermediate/saudi_arabia/osm/highways.geojson")
# prm_subway_df = gpd.read_file("../data/02_intermediate/saudi_arabia/osm/subway.geojson")

In [25]:
dataframesList = [prm_poi_df, prm_highways_df, prm_subway_df]

In [26]:
prm_geospatial = gpd.GeoDataFrame( pd.concat( dataframesList, ignore_index=True) )

In [27]:
prm_geospatial.head()

,amenity,amenity_map,geometry,highway,lanes,highway_map,railway,station
0,parking,transportation,POINT (46.70295 24.67299),NaN,NaN,NaN,NaN,NaN
1,hospital,hospitals,POINT (50.20046 26.30361),NaN,NaN,NaN,NaN,NaN
2,fire_station,community,POINT (50.12658 26.30858),NaN,NaN,NaN,NaN,NaN
3,cinema,entertainment,POINT (50.12946 26.30378),NaN,NaN,NaN,NaN,NaN
4,restaurant,restaurants,POINT (50.09195 26.29667),NaN,NaN,NaN,NaN,NaN


In [30]:
prm_geospatial.shape

(410278, 8)

In [ ]:
# prm_geospatial.to_file("../data/03_primary/saudi_arabia/osm/prm_geospatial.geojson", driver="GeoJSON")  

In [ ]:
# prm_geospatial = gpd.read_file("../data/03_primary/saudi_arabia/osm/prm_geospatial.geojson")

In [31]:
# prm_poi_df["centroid"] = prm_poi_df.geometry.centroid
prm_geospatial["latitude"] = prm_geospatial.geometry.centroid.y
prm_geospatial["longitude"] = prm_geospatial.geometry.centroid.x

/var/folders/3r/sqxygt_d4gs0bc8yhzxh19vw0000gp/T/ipykernel_11783/1610864879.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  prm_geospatial["latitude"] = prm_geospatial.geometry.centroid.y
/var/folders/3r/sqxygt_d4gs0bc8yhzxh19vw0000gp/T/ipykernel_11783/1610864879.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  prm_geospatial["longitude"] = prm_geospatial.geometry.centroid.x


In [32]:
geospatial = prm_geospatial

# Branches

In [33]:
prm_branches = pd.read_parquet("../data/03_primary/prm_branches", engine="fastparquet")

In [40]:
prm_branches[prm_branches["branch_id"].str.contains("BF088D263AB1")]

,branch_id,longitude,latitude,branch_code,branch_type,is_active,city
266,46490051-5A49-40CA-AAE8-BF088D263AB1,35.93,31.92,WS11,None,True,JEDDAH


In [34]:
prm_branches_info = prm_branches[["branch_id", "latitude", "longitude"]].copy()
prm_branches_info.head()

,branch_id,latitude,longitude
0,003B2F44-3384-4D43-A822-DB011B552F70,NaN,NaN
1,006D8F31-428E-4FFB-98D4-9C3AE2F1B4EE,25.360001,49.540001
2,00A68371-05D0-4B48-82E3-811740562799,24.793606,46.634247
3,017D919B-C594-487B-8E01-9E29E8B117EF,NaN,NaN
4,01D07F68-67A3-42F1-B2E5-694028D89272,21.568890,39.169998


In [35]:
prm_branches_info.shape

(958, 3)

In [36]:
print(prm_branches_info["latitude"].min(), prm_branches_info["latitude"].max())
print(prm_branches_info["longitude"].min(), prm_branches_info["longitude"].max())

0.0 31.92
0.0 50.21852


In [39]:
prm_branches_info[prm_branches_info["latitude"] == prm_branches_info["latitude"].max()]

,branch_id,latitude,longitude,latitude_rad,longitude_rad
266,46490051-5A49-40CA-AAE8-BF088D263AB1,31.92,35.93,0.557109,0.627097


In [37]:
prm_branches_info["latitude_rad"] = prm_branches_info["latitude"].astype(float).apply(radians)
prm_branches_info["longitude_rad"] = prm_branches_info["longitude"].astype(float).apply(radians)
prm_branches_info = prm_branches_info.dropna(axis=0, how="any")

In [38]:
print(geospatial["latitude"].min(), geospatial["latitude"].max())
print(geospatial["longitude"].min(), geospatial["longitude"].max())

16.73900210430301 28.581998511775247
36.41055679321289 50.352240063720075


In [23]:
geospatial

,amenity,amenity_map,geometry,highway,lanes,highway_map,railway,station,latitude,longitude,latitude_rad,longitude_rad
0,parking,transportation,POINT (46.70295 24.67299),NaN,NaN,NaN,NaN,NaN,46.702946,24.672993,0.815120,0.430625
1,hospital,hospitals,POINT (50.20046 26.30361),NaN,NaN,NaN,NaN,NaN,50.200459,26.303614,0.876163,0.459085
2,fire_station,community,POINT (50.12658 26.30858),NaN,NaN,NaN,NaN,NaN,50.126575,26.308580,0.874874,0.459171
3,cinema,entertainment,POINT (50.12946 26.30378),NaN,NaN,NaN,NaN,NaN,50.129459,26.303778,0.874924,0.459088
4,restaurant,restaurants,POINT (50.09195 26.29667),NaN,NaN,NaN,NaN,NaN,50.091949,26.296671,0.874269,0.458963
...,...,...,...,...,...,...,...,...,...,...,...,...
410273,NaN,NaN,"POLYGON ((46.67574 24.7127, 46.67484 24.71437,...",NaN,NaN,NaN,station,subway,46.675143,24.713481,0.814635,0.431332
410274,NaN,NaN,"POLYGON ((46.67096 24.72174, 46.67115 24.72181...",NaN,NaN,NaN,station,subway,46.671338,24.721193,0.814569,0.431466
410275,NaN,NaN,"POLYGON ((46.71744 24.84189, 46.71743 24.84194...",NaN,NaN,NaN,station,None,46.717407,24.841571,0.815373,0.433567
410276,NaN,NaN,"POLYGON ((46.61533 24.83142, 46.61475 24.83116...",NaN,NaN,NaN,station,subway,46.615412,24.830468,0.813592,0.433373


In [18]:
geospatial["latitude_rad"] = geospatial["latitude"].astype(float).apply(radians)
geospatial["longitude_rad"] = geospatial["longitude"].astype(float).apply(radians)

ball = BallTree(geospatial[["latitude_rad", "longitude_rad"]].values, metric="haversine")

In [19]:
radius_list = [300, 500, 1000]
max_radius = np.max(radius_list)

In [20]:

indices, distances = ball.query_radius(
    prm_branches_info[["latitude_rad", "longitude_rad"]].values, 
    r=max_radius/6371000,
    return_distance=True,
)
distances = distances * 6371

prm_branches_info["geo_point_index"] = indices
prm_branches_info["distances_osm"] = distances


In [21]:
indices

array([array([], dtype=int64), array([], dtype=int64),
       array([], dtype=int64), array([], dtype=int64),
       array([], dtype=int64), array([], dtype=int64),
       array([], dtype=int64), array([], dtype=int64),
       array([], dtype=int64), array([], dtype=int64),
       array([], dtype=int64), array([], dtype=int64),
       array([], dtype=int64), array([], dtype=int64),
       array([], dtype=int64), array([], dtype=int64),
       array([], dtype=int64), array([], dtype=int64),
       array([], dtype=int64), array([], dtype=int64),
       array([], dtype=int64), array([], dtype=int64),
       array([], dtype=int64), array([], dtype=int64),
       array([], dtype=int64), array([], dtype=int64),
       array([], dtype=int64), array([], dtype=int64),
       array([], dtype=int64), array([], dtype=int64),
       array([], dtype=int64), array([], dtype=int64),
       array([], dtype=int64), array([], dtype=int64),
       array([], dtype=int64), array([], dtype=int64),
       arr

In [196]:
prm_branches_info_exploded = prm_branches_info.explode(["geo_point_index", "distances_osm"])

In [197]:
for radius in radius_list:
    prm_branches_info_exploded[f"is_within_{radius:04}"] = np.where(
        prm_branches_info_exploded["distances_osm"] <= radius/1000,
        1,
        0
    )

In [199]:

prm_branches_info_geo_info = prm_branches_info_exploded.merge(
    geospatial.drop(columns=["latitude_rad", "longitude_rad", "latitude", "longitude", "geometry"]),
    right_index=True,
    left_on="geo_point_index",
    how="left",
)

In [200]:
prm_branches_info_geo_info.shape, prm_branches_info_geo_info.drop_duplicates().shape

((122681, 17), (122681, 17))

In [201]:
prm_branches_info_geo_info.head()

,branch_id,latitude,longitude,latitude_rad,longitude_rad,geo_point_index,distances_osm,is_within_0300,is_within_0500,is_within_1000,amenity,amenity_map,highway,lanes,highway_map,railway,station
1,006D8F31-428E-4FFB-98D4-9C3AE2F1B4EE,49.540001,25.360001,0.864636,0.442616,NaN,NaN,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00A68371-05D0-4B48-82E3-811740562799,46.634247,24.793606,0.813921,0.432730,130677,0.955848,0,0,1,None,None,tertiary_link,None,highway,None,None
2,00A68371-05D0-4B48-82E3-811740562799,46.634247,24.793606,0.813921,0.432730,91552,0.948401,0,0,1,None,None,tertiary,None,highway,None,None
2,00A68371-05D0-4B48-82E3-811740562799,46.634247,24.793606,0.813921,0.432730,8831,0.850149,0,0,1,place_of_worship,community,None,None,None,None,None
2,00A68371-05D0-4B48-82E3-811740562799,46.634247,24.793606,0.813921,0.432730,68803,0.824046,0,0,1,None,None,living_street,None,None,None,None


In [208]:
agg_dict = {
    f"n_places_within_{radius:04}m": (f"is_within_{radius:04}", "sum")
    for radius in radius_list
}

In [209]:
ftr_geospatial_agg = prm_branches_info_geo_info.groupby(["branch_id", "amenity_map"], as_index=False).agg(
    **agg_dict
)

ftr_geospatial_osm = ftr_geospatial_agg.pivot_table(
    index="branch_id", columns="amenity_map", fill_value=0, margins=True
)

In [210]:
ftr_geospatial_osm.columns = ["__".join(col).strip() for col in ftr_geospatial_osm.columns]
ftr_geospatial_osm = ftr_geospatial_osm.reset_index()

In [211]:
ftr_geospatial_osm.head()

,branch_id,n_places_within_0300m__community,n_places_within_0300m__educational_establishment,n_places_within_0300m__entertainment,n_places_within_0300m__facilities,n_places_within_0300m__hospitals,n_places_within_0300m__marketplaces,n_places_within_0300m__payments,n_places_within_0300m__restaurants,n_places_within_0300m__transportation,...,n_places_within_1000m__community,n_places_within_1000m__educational_establishment,n_places_within_1000m__entertainment,n_places_within_1000m__facilities,n_places_within_1000m__hospitals,n_places_within_1000m__marketplaces,n_places_within_1000m__payments,n_places_within_1000m__restaurants,n_places_within_1000m__transportation,n_places_within_1000m__All
0,00A68371-05D0-4B48-82E3-811740562799,1.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,1.0,...,14.0,5.0,1.0,0.0,2.0,0.0,6.0,18.0,14.0,8.571429
1,01D07F68-67A3-42F1-B2E5-694028D89272,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,7.0,...,10.0,6.0,0.0,0.0,1.0,8.0,5.0,12.0,24.0,9.428571
2,01E30C30-B4B6-4087-B994-1509989401AD,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,14.0,3.0,0.0,0.0,0.0,0.0,0.0,2.0,3.0,5.500000
3,01E52D08-5F0C-49EA-B05C-A9AE8CAF5F6B,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.000000
4,01E7599A-E311-45B2-BA7D-15BF3AE149DC,3.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,1.0,...,16.0,0.0,0.0,0.0,0.0,8.0,13.0,28.0,16.0,16.200000


In [212]:
(
    prm_branches_info_geo_info["branch_id"].nunique(),
    prm_branches_info_geo_info.dropna(subset="amenity_map")["branch_id"].nunique(),
    ftr_geospatial_osm["branch_id"].nunique(),
)

(827, 346, 347)

In [213]:
world_pop["latitude_rad"] = world_pop["latitude"].astype(float).apply(radians)
world_pop["longitude_rad"] = world_pop["longitude"].astype(float).apply(radians)
ball = BallTree(
    world_pop[["latitude_rad", "longitude_rad"]].values,
    metric="haversine",
)
distances, indices = ball.query(prm_branches_info[["latitude_rad", "longitude_rad"]].values, k=1)
distances = distances * 6371

prm_branches_info["geo_point_index"] = [i[0] for i in indices]
ftr_geospatial_worldpop = prm_branches_info.merge(
    world_pop.drop(columns=["latitude_rad", "longitude_rad", "latitude", "longitude"]),
    right_index=True,
    left_on="geo_point_index",
    how="left",
)[["branch_id", "pop_density"]]


In [214]:
ftr_geospatial_worldpop.head()

,branch_id,pop_density
1,006D8F31-428E-4FFB-98D4-9C3AE2F1B4EE,408.754425
2,00A68371-05D0-4B48-82E3-811740562799,2479.327393
4,01D07F68-67A3-42F1-B2E5-694028D89272,2192.117188
5,01E30C30-B4B6-4087-B994-1509989401AD,2629.684814
6,01E52D08-5F0C-49EA-B05C-A9AE8CAF5F6B,2289.483154


In [215]:
ftr_geospatial_worldpop.shape

(827, 2)

In [216]:
ftr_geospatial = prm_branches.merge(
    ftr_geospatial_osm,
    on="branch_id",
    how="left"
).merge(
    ftr_geospatial_worldpop,
    on="branch_id",
    how="left"
)

In [217]:
ftr_geospatial

,branch_id,longitude,latitude,branch_code,branch_type,is_active,city,n_places_within_0300m__community,n_places_within_0300m__educational_establishment,n_places_within_0300m__entertainment,...,n_places_within_1000m__educational_establishment,n_places_within_1000m__entertainment,n_places_within_1000m__facilities,n_places_within_1000m__hospitals,n_places_within_1000m__marketplaces,n_places_within_1000m__payments,n_places_within_1000m__restaurants,n_places_within_1000m__transportation,n_places_within_1000m__All,pop_density
0,003B2F44-3384-4D43-A822-DB011B552F70,NaN,NaN,2280,None,True,AL SULAYIL,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,006D8F31-428E-4FFB-98D4-9C3AE2F1B4EE,25.360001,49.540001,1206,PE,True,AL HASSA,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,408.754425
2,00A68371-05D0-4B48-82E3-811740562799,24.793606,46.634247,1838,PE,False,RIYADH,1.0,0.0,0.0,...,5.0,1.0,0.0,2.0,0.0,6.0,18.0,14.0,8.571429,2479.327393
3,017D919B-C594-487B-8E01-9E29E8B117EF,NaN,NaN,129,None,False,JEDDAH,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,01D07F68-67A3-42F1-B2E5-694028D89272,21.568890,39.169998,1023,PE,True,JEDDAH,0.0,0.0,0.0,...,6.0,0.0,0.0,1.0,8.0,5.0,12.0,24.0,9.428571,2192.117188
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
953,FF0B0C86-1E9A-4671-953E-A75327302A36,26.393612,50.070557,1990,PE,True,DAMMAM,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2603.124268
954,FF0D1E8D-B3B8-4C13-80F8-6EFEFC42BD92,0.000000,0.000000,2221,None,True,AL KHOBAR,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.670512
955,FF29F235-85D9-4FBC-B1C3-79B39D6D8DBB,24.384722,39.614723,1904,PE,True,MADINA,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.000000,924.338013
956,FFBCD6BF-8787-4E57-945E-16A0A74963BC,24.158890,47.324165,1858,PE,True,AL KHARJ,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,331.071899


In [2]:
prm_geolocation = pd.read_parquet("../data/04_feature/ftr_geolocation")

In [18]:
prm_geolocation[prm_geolocation["branch_code"] == "1838"]

,branch_id,longitude,latitude,branch_code,branch_type,is_active,city,n_places_within_1000__community,n_places_within_1000__educational_establishment,n_places_within_1000__entertainment,...,n_places_within_500__educational_establishment,n_places_within_500__entertainment,n_places_within_500__facilities,n_places_within_500__hospitals,n_places_within_500__marketplaces,n_places_within_500__payments,n_places_within_500__restaurants,n_places_within_500__transportation,n_places_within_500__All,pop_density
2,00A68371-05D0-4B48-82E3-811740562799,24.793606,46.634247,1838,PE,False,RIYADH,14.0,5.0,1.0,...,1.0,1.0,0.0,0.0,0.0,0.0,5.0,3.0,2.0,2479.327393
3,00A68371-05D0-4B48-82E3-811740562799,24.793606,46.634247,1838,PE,False,RIYADH,14.0,5.0,1.0,...,1.0,1.0,0.0,0.0,0.0,0.0,5.0,3.0,2.0,2479.327393
4,00A68371-05D0-4B48-82E3-811740562799,24.793606,46.634247,1838,PE,False,RIYADH,14.0,5.0,1.0,...,1.0,1.0,0.0,0.0,0.0,0.0,5.0,3.0,2.0,2479.327393
5,00A68371-05D0-4B48-82E3-811740562799,24.793606,46.634247,1838,PE,False,RIYADH,14.0,5.0,1.0,...,1.0,1.0,0.0,0.0,0.0,0.0,5.0,3.0,2.0,2479.327393
6,00A68371-05D0-4B48-82E3-811740562799,24.793606,46.634247,1838,PE,False,RIYADH,14.0,5.0,1.0,...,1.0,1.0,0.0,0.0,0.0,0.0,5.0,3.0,2.0,2479.327393
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
451,00A68371-05D0-4B48-82E3-811740562799,24.793606,46.634247,1838,PE,False,RIYADH,14.0,5.0,1.0,...,1.0,1.0,0.0,0.0,0.0,0.0,5.0,3.0,2.0,2479.327393
452,00A68371-05D0-4B48-82E3-811740562799,24.793606,46.634247,1838,PE,False,RIYADH,14.0,5.0,1.0,...,1.0,1.0,0.0,0.0,0.0,0.0,5.0,3.0,2.0,2479.327393
453,00A68371-05D0-4B48-82E3-811740562799,24.793606,46.634247,1838,PE,False,RIYADH,14.0,5.0,1.0,...,1.0,1.0,0.0,0.0,0.0,0.0,5.0,3.0,2.0,2479.327393
454,00A68371-05D0-4B48-82E3-811740562799,24.793606,46.634247,1838,PE,False,RIYADH,14.0,5.0,1.0,...,1.0,1.0,0.0,0.0,0.0,0.0,5.0,3.0,2.0,2479.327393


In [5]:
cols = ['branch_id', 'longitude', 'latitude', 'branch_code', 'branch_type', 'is_active', 'city', 'n_places_within_300__All', 'n_places_within_500__All', 'n_places_within_1000__All', 'pop_density']

In [6]:
# prm_geolocation.columns

In [7]:
prm_geolocation[cols]

,branch_id,longitude,latitude,branch_code,branch_type,is_active,city,n_places_within_300__All,n_places_within_500__All,n_places_within_1000__All,pop_density
0,003B2F44-3384-4D43-A822-DB011B552F70,NaN,NaN,2280,None,True,AL SULAYIL,NaN,NaN,NaN,NaN
1,006D8F31-428E-4FFB-98D4-9C3AE2F1B4EE,25.360001,49.540001,1206,PE,True,AL HASSA,NaN,NaN,NaN,408.754425
2,00A68371-05D0-4B48-82E3-811740562799,24.793606,46.634247,1838,PE,False,RIYADH,0.857143,2.0,8.571429,2479.327393
3,00A68371-05D0-4B48-82E3-811740562799,24.793606,46.634247,1838,PE,False,RIYADH,0.857143,2.0,8.571429,2479.327393
4,00A68371-05D0-4B48-82E3-811740562799,24.793606,46.634247,1838,PE,False,RIYADH,0.857143,2.0,8.571429,2479.327393
...,...,...,...,...,...,...,...,...,...,...,...
122807,FF29F235-85D9-4FBC-B1C3-79B39D6D8DBB,24.384722,39.614723,1904,PE,True,MADINA,0.000000,0.0,1.000000,924.338013
122808,FF29F235-85D9-4FBC-B1C3-79B39D6D8DBB,24.384722,39.614723,1904,PE,True,MADINA,0.000000,0.0,1.000000,924.338013
122809,FF29F235-85D9-4FBC-B1C3-79B39D6D8DBB,24.384722,39.614723,1904,PE,True,MADINA,0.000000,0.0,1.000000,924.338013
122810,FFBCD6BF-8787-4E57-945E-16A0A74963BC,24.158890,47.324165,1858,PE,True,AL KHARJ,NaN,NaN,NaN,331.071899


In [12]:
prm_geolocation.drop_duplicates()[cols]

,branch_id,longitude,latitude,branch_code,branch_type,is_active,city,n_places_within_300__All,n_places_within_500__All,n_places_within_1000__All,pop_density
0,003B2F44-3384-4D43-A822-DB011B552F70,NaN,NaN,2280,None,True,AL SULAYIL,NaN,NaN,NaN,NaN
1,006D8F31-428E-4FFB-98D4-9C3AE2F1B4EE,25.360001,49.540001,1206,PE,True,AL HASSA,NaN,NaN,NaN,408.754425
2,00A68371-05D0-4B48-82E3-811740562799,24.793606,46.634247,1838,PE,False,RIYADH,0.857143,2.0,8.571429,2479.327393
456,017D919B-C594-487B-8E01-9E29E8B117EF,NaN,NaN,129,None,False,JEDDAH,NaN,NaN,NaN,NaN
457,01D07F68-67A3-42F1-B2E5-694028D89272,21.568890,39.169998,1023,PE,True,JEDDAH,1.428571,2.0,9.428571,2192.117188
...,...,...,...,...,...,...,...,...,...,...,...
122567,FF0B0C86-1E9A-4671-953E-A75327302A36,26.393612,50.070557,1990,PE,True,DAMMAM,NaN,NaN,NaN,2603.124268
122568,FF0D1E8D-B3B8-4C13-80F8-6EFEFC42BD92,0.000000,0.000000,2221,None,True,AL KHOBAR,NaN,NaN,NaN,11.670512
122569,FF29F235-85D9-4FBC-B1C3-79B39D6D8DBB,24.384722,39.614723,1904,PE,True,MADINA,0.000000,0.0,1.000000,924.338013
122810,FFBCD6BF-8787-4E57-945E-16A0A74963BC,24.158890,47.324165,1858,PE,True,AL KHARJ,NaN,NaN,NaN,331.071899


In [14]:
prm_geolocation.drop_duplicates()[cols].to_excel("branch_geodata.xlsx", index=False, engine="openpyxl")